# Практикум по программированию на языке Python

## <font color='lilac'>Занятие 6: Основы ООП: типизация и полиморфизм, классы данных, декораторы</font>

#### Роман Ищенко (roman.ischenko@gmail.com), Мурат Апишев (mel-lain@yandex.ru)
#### Москва, 2026

### <font color='lilac'>Напоминание: принципы ООП</font>

- **Абстракция** - выделение важных свойств объекта и игнорирование прочих
- **Инкапсуляция** - хранение данных и методов работы с ними внутри одного класса с доступом к данным только через методы
- **Наследование** - возможность создания наследников, получающих все свойства родителей с возможностью их переопределения и расширения
- **Полиморфизм** - возможность использования объектов разных типов с общим интерфейсом без информации об их внутреннем устройстве

### <font color='lilac'>Напоминание: класс, объект, интерфейс</font>

- __Класс__ представляет собой тип данных (как int или str)
- Это способ описания некоторой сущности, её состояния и возможного поведения
- Поведение при этом зависит от состояния и может его изменять

- __Объект__ - это конретный представитель класса (как переменная этого типа)
- У объекта своё состояние, изменяемое поведением
- Поведение полностью определяется правилами, описанными в классе

- __Интерфейс__ - это класс, описывающий только поведение, без состояния
- Создать объект типа интерфейса невозможно (если есть их поддержка на уровне языка)
- Поведение полностью определяется правилами, описанными в классе
- Вместо этого описываются классы, которые реализуют этот интерфейс и, в то же время, имеют состояние

### <font color='lilac'>Абстрактные классы</font>

- Промежуточное состояние между чистым интерфейсом и полноценным классом
- Эмулировать абстрактные классы можно с помощью методов, которые бросают исключение в своей реализации по-умолчанию:

In [ ]:
class Abstract1:
    def method(self):
        raise NotImplementedError

#Abstract1().method() -> NotImplementedError

- Подход плох тем, что возможность создать объект класса всё равно сохраняется и ошибка произойдёт в момент обращения к методу
- Альтернатива - класс `ABC` (на самом деле его метакласс `ABCMeta`) и декораторы `abstractmethod` из библиотеки `abc`:

In [1]:
from abc import ABC, abstractmethod

class Abstract(ABC):
    @abstractmethod
    def my_abstract_method(self):
        ...

#Abstract() -> TypeError: Can't instantiate abstract class Abstract with abstract method my_abstract_method

class Cls(Abstract):
    def my_abstract_method(self):
        ...

Cls()

### <font color='lilac'>Абстрактные классы</font>

- Для версий Python ниже 3.8 есть различные версии декораторов (`@abstractstaticmethod`, `@abstractproperty`, `@abstractclassmethod`)
- В новых версиях язык поддерживает комбинирование декораторов (`@abstractmethod` должен находиться внутри):

In [2]:
class Abstract(ABC):
    @staticmethod
    @abstractmethod
    def my_abstract_staticmethod():
        ...

    @classmethod
    @abstractmethod
    def my_abstract_classmethod(cls):
        ...

    @property
    @abstractmethod
    def my_abstract_property(self):
        ...

    @my_abstract_property.setter
    @abstractmethod
    def my_abstract_property(self, val):
        ...

### <font color='lilac'>Аннотация типов</font>

- Последний принцип ООП, который требуется рассмотреть - полиморфизм
- Для обсуждение интерфейсов и полиморфизма нужно детальнее рассмотреть аннотирование типов и статическую проверку типизации
- До появления аннотирования понятие интерфейса в Python не имело особого смысла

- __Виды типизации__ (одна из классификаций):

    1. Утиная (duck-typing)
    2. Номинальная (nominal type system)
    3. Структурная (structural type system)

### <font color='lilac'>Подход 1: утиная типизация</font>

- "Если что-то ведёт себя как утка, значит это - утка"
- Эта концепция возникает в языках с динамической типизацией (Python, JavaScript) и означает, что при использовании объекта
    - его конкретный класс не имеет значения
    - важны его атрибуты (поля и методы)
- Т.е. объект принимается без каких-либо проверок, и если он имеет нужные атрибуты - код выполнится корректно, если не имеет - нет

In [10]:
class A:
    def __eq__(self, val):
        return True

# The object with __eq__ method is expected
print(A() == 3)

True


- Пример выше показывает, что утиная типизация при отсутствии контроля типов может привести к неприятным последствиям
- На утиной типизации основан механизм _полиморфизма_ в Python

### <font color='lilac'>Полиморфизм</font>

- Полиморфизм позволяет работать с объектами, основываясь только на их интерфейсе, без знания типа
- В C++ требуется, чтобы объекты полиморфных классов имели общего предка
- В общем случае в Python это необязательно, достаточно, чтобы объекты поддерживали один интерфейс (duck-typing)
- Пример:

In [11]:
class Square:
    def __init__(self, side):
        self.side = side
    
    def area(self):
        return self.side ** 2

class Triangle:
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c

    def area(self):
        s = (self.a + self.b + self.c) / 2.0
        return (s *(s - self.a) * (s - self.b) * (s - self.c)) ** 0.5

In [12]:
def compute_areas(figures):
    for figure in figures:
        print(figure.area(), end=' ')

- Можно запускать, не беспокоясь о том, что именно представляют собой входные объекты:

In [13]:
compute_areas([Square(10), Triangle(1, 3, 3)])

100 1.479019945774904 

### <font color='lilac'>Подход 2: номинальная типизация</font>

- Совместимость типов определяется через явные декларации в коде (имена типов и иерархия наследования)
- Такой подход используется повсеместно в языках со статической типизацией (C++, Java)
- В случае Python этого можно добиться с помощью статической проверки

In [15]:
class Bird:
    def feed(self) -> None: print('OK')

class Duck(Bird):
    def feed(self) -> None: print('OK')

class Goose:
    def feed(self) -> None: print('OK')

def feed(bird: Bird) -> None:
    bird.feed()

feed(Bird())
feed(Duck())
#feed(Goose()) -> static check error (OK in runtime due to duck typing)

OK
OK
OK


### <font color='lilac'>Напоминание: аннотации типов</font>

- Ранее были рассмотрены базовые примеры аннотаций для скалярных и контейнерных (generic) типов:

In [16]:
from typing import Set

def print_scalar(obj: int) -> None:
    print(obj)

def print_set(obj: Set[int]) -> None:
    print(obj)

- Возможности `typing` существенно шире. Например, он
    - вводит новые типы (Union и т.п.)
    - позволяет определять собственные generic-классы


- Но для полноценного использования и изучения сперва нужно добавить статическую проверку типов

### <font color='lilac'>Модуль MyPy: статическая проверка типов</font>

- Модуль `mypy` - стандартный инструмент для статической проверки аннотированного кода на Python (ещё есть `pytype` и `pyright`)
- Установка и использование стандартные:

In [17]:
#!pip install mypy

def check_last():
    with open('temp.txt', 'w') as fout:
        fout.write(In[len(In)-2])
    !mypy temp.txt

In [18]:
def func(n: int = 10) -> int:
    return n ** 2

func(2.5)

6.25

In [19]:
check_last()

temp.txt:4: error: Argument 1 to "func" has incompatible type "float"; expected "int"  [arg-type]
Found 1 error in 1 file (checked 1 source file)


### <font color='lilac'>Тип Union</font>

In [22]:
from typing import Union, List, Set

def func(x: Union[List[int], Set[str]]) -> None:
    ...

func([1, 2, 3])
func({1, 2, '3'})
func({'s', 't', 'r'})

In [23]:
check_last()

temp.txt:7: error: Argument 1 to <set> has incompatible type "int"; expected "str"  [arg-type]
temp.txt:7: error: Argument 2 to <set> has incompatible type "int"; expected "str"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)


### <font color='lilac'>Тип Any</font>

- Тип `Any` говорит о том, что в этом месте может быть произвольный тип, и проверка кода игнорирует все, связанное с переменной типа `Any`
- Для `Any` верны следующие утверждения:
    - любой объект является объектом типа `Any`
    - любой класс является подклассом типа `Any`
- Несмотря на схожесть, `Any` и `object` - это не одно и то же - тип `object` ограничивает множество операций теми, что допускает `object`, а `Any` допускает всё:

In [26]:
from typing import Any

def func_any(x: Any) -> None: x.nothing()
def func_object(x: object) -> None: x.nothing()

# func_any(None)
# func_object(None)

In [27]:
check_last()

temp.txt:4: error: "object" has no attribute "nothing"  [attr-defined]
Found 1 error in 1 file (checked 1 source file)


### <font color='lilac'>Тип Optional</font>

In [28]:
from typing import Optional, List

def func(x: Optional[List[int]]) -> None:
    ...

func([1, 2, 3])
func(None)

In [29]:
check_last()

Success: no issues found in 1 source file


В Python 3.10+ допускается синтаксис с "|" (в то же смысле, что и в TypeScript):

In [30]:
def func(x: list[int] | None) -> None:
    ...

func([1, 2, 3])
func(None)

In [31]:
check_last()

Success: no issues found in 1 source file


### <font color='lilac'>Тип Literal</font>

- Тип `Literal` параметризуется не другим типом, а конкретным значением

In [32]:
from typing import Literal

def func(x: Literal[3], y: Literal['something']) -> None:
    ...

func(3, 'something')
func(4, 'nothing')

In [33]:
check_last()

temp.txt:7: error: Argument 1 to "func" has incompatible type "Literal[4]"; expected "Literal[3]"  [arg-type]
temp.txt:7: error: Argument 2 to "func" has incompatible type "Literal['nothing']"; expected "Literal['something']"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)


### <font color='lilac'>Определение собственных generic типов</font>

- Функция `TypeVar` позволяет получить ссылку на тип переменной, имя которой было подано в качестве параметра
- Ссылка на тип параметра позволяет создавать новые generic типы:

In [34]:
from typing import TypeVar, Generic, List

T = TypeVar('T')

class Stack(Generic[T]):
    def __init__(self) -> None:
        self.items: List[T] = []  

    def push(self, item: T) -> None:
        self.items.append(item)

stack: Stack[int] = Stack()
stack.push(10)
stack.push('10')

In [35]:
check_last()

temp.txt:14: error: Argument 1 to "push" of "Stack" has incompatible type "str"; expected "int"  [arg-type]
Found 1 error in 1 file (checked 1 source file)


### <font color='lilac'>Подход 3: структурная типизация</font>

- Совместимость типов определяется на основе структуры типов, а не на явных декларациях
- Этот подход аналогичен утиной типизации за исключением того, что проверка является статической, а не динамической, это обеспечивает корректность типов без необходимости явных наследований, что повышает гибкость кода и независимость модулей и классов
- Структурная типизация используется в TypeScript и Go

- Python допускает использование структурной типизации с версии 3.8, для чего требуются аннотирование типов и _протоколы_
- Протокол по сути является интерфейсом, которому объект должен удовлетворять для совместимости типов
- В модуле `typing` есть много стандартных протоколов, например:
    - Mapping
    - Iterable
    - Callable
    - Hashable
    - Reversible
    - ...

### <font color='lilac'>Примеры использования протоколов</font>

- Протокол `Callable` требует наличия у реализации интерфейса метода `__call__`
- Протокол `Mapping` требует наличия у реализации интерфейса метода `__getitem__`

In [36]:
from typing import Callable

def func(f: Callable[[int, int], bool]) -> bool:
    return f(1, 2)

func(lambda x, y: x == y)

False

In [37]:
from typing import Mapping

def func(m: Mapping[str, int], key: str) -> int:
    return m[key]

func({'k': 0}, 'k')

0

### <font color='lilac'>Определение собственного протокола</font>

- Этот и следующий примеры взяты из [статьи](https://habr.com/ru/post/557898)
- Протокол и корректная его реализация:

In [45]:
from typing import Protocol

class Figure(Protocol):
    name: str

    def calculate_area(self) -> float: pass

    def calculate_perimeter(self) -> float: pass

def show(figure: Figure) -> None:
    print(f'S ({figure.name}) = {figure.calculate_area()}')
    print(f'P ({figure.name}) = {figure.calculate_perimeter()}')


class Square:
    name = 'square'

    def __init__(self, size: float): self.size = size
    def calculate_area(self) -> float: return self.size * self.size
    def calculate_perimeter(self) -> float: return 4 * self.size
        
    def set_color(self, color: str) -> None: self.color = color

show(Square(size=3.14))

S (square) = 9.8596
P (square) = 12.56


In [46]:
check_last()

Success: no issues found in 1 source file


### <font color='lilac'>Определение собственного протокола</font>

- Некорректная реализация и детекция ошибки на этапе статической проверки:

In [48]:
from typing import Protocol

class Figure(Protocol):
    name: str

    def calculate_area(self) -> float: pass

    def calculate_perimeter(self) -> float: pass

def show(figure: Figure) -> None:
    # print(f'S ({figure.name}) = {figure.calculate_area()}')
    print(f'P ({figure.name}) = {figure.calculate_perimeter()}')


class Circle:
    PI = 3.1415926
    name = "Circle"

    def __init__(self, radius: float):
        self.radius = radius

    def calculate_perimeter(self) -> float:
        return 2 * self.PI * self.radius

show(Circle(radius=1))

P (Circle) = 6.2831852


In [49]:
check_last()

temp.txt:25: error: Argument 1 to "show" has incompatible type "Circle"; expected "Figure"  [arg-type]
temp.txt:25: note: "Circle" is missing following "Figure" protocol member:
temp.txt:25: note:     calculate_area
Found 1 error in 1 file (checked 1 source file)


### <font color='lilac'>Forward references</font>

- Иногда в коде возникает необходимость сослаться на тип, который ещё не был определён

In [50]:
class Foo:
    def bar(self) -> Foo:
        return Foo()

# NameError: name 'Foo' is not defined

NameError: name 'Foo' is not defined

In [51]:
class Foo:
    def bar(self) -> Bar:
        return Bar()

class Bar:
    def foo(self) -> Foo:
        return Foo()

# NameError: name 'Bar' is not defined

NameError: name 'Bar' is not defined

### <font color='lilac'>Forward references</font>

- Решение - использовать строковое представление имени типа вместо самого типа:

In [52]:
class Foo:
    def bar(self) -> 'Foo':
        return Foo()

In [53]:
class Foo:
    def bar(self) -> 'Bar':
        return Bar()

class Bar:
    def foo(self) -> Foo:
        return Foo()

- Начиная с Python 3.7 заботу об этом может взять на себя импорт `from __future__ import annotations`
- Он автоматически производит замену всех типов на имена-строки

In [54]:
from __future__ import annotations

class Foo:
    def bar(self) -> Foo:
        return Foo()

### <font color='lilac'>Хранение типизированных данных</font>

- Хранение типизированных данных требует строгости и аккуратности
- В простейших случаях можно воспользоваться словарём или кортежем:

In [ ]:
tuple_data = (0, 'string')
dict_data = {'int_field': 0, 'str_field': 'string'}

- отсутствие именованного типа может привести к ошибке, нужно помнить, что переменная ссылается на нужную стркутуру
- необходимо следить за ключами словаря и порядком аргументов кортежа, это автоматически не проверяется

- Возможный вариант - _именованные кортежи_ (Namedtuple):

In [56]:
from collections import namedtuple

Data = namedtuple('Data', ['int_field', 'str_field'])
named_tuple_data = Data(0, 'string')
named_tuple_data

Data(int_field=0, str_field='string')

- При таком подходе:
    - кортежи являются неизменяемыми
    - по кортежам можно итерироваться
- Минус
    - кортежи с разными наборами ключей, но одинаковыми значениями будут считаться одинаковыми

In [57]:
Data2 = namedtuple('Data2', ['int_field_2', 'str_field_2'])

print(*named_tuple_data)
print(Data2(0, 'string') == named_tuple_data)

0 string
True


### <font color='lilac'>Классы данных</font>

- Эти проблемы решаются созданием для данных отдельного типа
- Пример типичного класса данных, описанного стандартными средствами языка:

In [ ]:
class Data:
    def __init__(self, int_field: int, str_field: str):
        self.int_field = int_field
        self.str_field = str_field

    def __repr__(self):
        return f"{self.__class__.__name__}(int_field={self.int_field:d}, str_field='{self.str_field:s}')"

    def __str__(self):
        return self.__repr__()

    def __eq__(self, other):
        return self.int_field == other.int_field and self.str_field == other.str_field

- Многие вещи выглядят типовыми и повторяющимися

### <font color='lilac'>Классы данных</font>

- Для решения проблем хранения типизированных данных в Python 3.7 были добавлены _классы данных_
- Этот механизм позволяет автоматически генерировать классы с типизированными полями со значениями по-умолчанию
- Пример того же класса данных, описанного с помощью декоратора `@dataclass`:

In [58]:
from dataclasses import dataclass

@dataclass
class Data:
    int_field: int
    str_field: str

- Этот класс обладает аналогичной функциональностью
- Аннотации типов в `dataclass` обязательны, поля без типов будут проигнорированы
- Полям можно задавать значения по-умолчанию:

In [ ]:
@dataclass
class Data:
    int_field: int = 10
    str_field: str = 'string'

Data()

- Проверка типов не производится, ошибки можно отловить только с помощью статического анализатора:

In [59]:
from dataclasses import dataclass

@dataclass
class Data:
    int_field: int = 'string'
    str_field: str = 10

Data()

Data(int_field='string', str_field=10)

In [60]:
check_last()

temp.txt:5: error: Incompatible types in assignment (expression has type "str", variable has type "int")  [assignment]
temp.txt:6: error: Incompatible types in assignment (expression has type "int", variable has type "str")  [assignment]
Found 2 errors in 1 file (checked 1 source file)


### <font color='lilac'>Классы данных</font>

- Есть параметры для управление генерацией класса (все методы, определённые пользователем, перетирают реализации по-умолчанию):
    - `frozen` - сделать класс неизменяемым или нет (по-умолчанию `False`)
    - `init` - сгенерировать для класса метод `__init__` (по-умолчанию `True`)
    - `repr` - сгенерировать для класса метод `__repr__` (по-умолчанию `True`)
    - `eq` - сгенерировать для класса метод `__eq__`, сраниваются типы и значения как кортежи (по-умолчанию `True`)
    - `order` - сгенерировать методы `__lt__`, `__le__`, `__gt__` и `__ge__`, сравнивая значения как кортежи (по-умолчанию `False`)
    - `unsafe_hash` - использовать небезопасные поля в подсчёте хэша объекта в методе `__hash__` (по-умолчанию `False`)

In [61]:
@dataclass(frozen=True, order=True)
class Data:
    int_field: int = 10
    str_field: str = 'string'

#Data().int_field = 5 -> FrozenInstanceError: cannot assign to field 'int_field'

Data(10) < Data(20)

True

- Классы данных можно преобразовывать в словари и кортежи:

In [62]:
from dataclasses import asdict, astuple

print(asdict(Data(20)))
print(astuple(Data(20)))

{'int_field': 20, 'str_field': 'string'}
(20, 'string')


### <font color='lilac'>Создание изменяемых полей</font>

- Создавать поля с изменяемыми типами и значениями по-умолчанию не так просто, как константные

In [ ]:
@dataclass
class Data:
    list_field: List[int] = []

# -> ValueError: mutable default <class 'list'> for field list_field is not allowed: use default_factory

- Корректное решение проблемы подсказывается в сообщении об ошибке:

In [ ]:
from dataclasses import field

@dataclass
class Data:
    list_field: List[int] = field(default_factory=list)

### <font color='lilac'>Пост-инициализация</font>

- При создании объекта может потребоваться выполнить после `__init__` по-умолчанию некоторую логику, не переписывая весь метод
- Для этого можно определить метод `__post_init__`, который `__init__` (если он определён) всегда вызывает после себя
- Можно задавать параметры для пост-инициализации: это поля типа `InitVar`, которые передаются в `__post_init__`
- В остальном классе эти поля игнорируются

In [71]:
from dataclasses import InitVar

@dataclass
class Data:
    float_field: float
    int_field: InitVar[int]

    def __post_init__(self, int_field: int) -> None:
        print(f'Created Data object with float value {self.float_field:.{int_field}f}')

d = Data(3.141562, 2)
#d.int_field -> AttributeError: 'Data' object has no attribute 'int_field'

Created Data object with float value 3.14


### <font color='lilac'>Наследование классов данных</font>

- Классы данных наследуются от `object` и могут использоваться для получения новых классов путём наследования
- Наследуемый класс тоже нужно помечать декоратором `@dataclass`
- Декоратор проходит по всем родительским классам (при множественном наследовании порядок MRO), для каждого класса поля сохраняются в упорядоченный словарь, остаются самые последние версии полей
- По этой причине, если поле в родительском классе определялось со значением по-умолчанию, то и в дочернем классе должно быть так (или останется старое значение)

In [69]:
@dataclass
class DataA:
    field_a: str
    field_b: str = 'default'

@dataclass
class DataB(DataA):
    field_b: str
    field_a: str = None
    field_c: int = None

DataB.__init__

<function __main__.DataB.__init__(self, field_a: 'str' = None, field_b: 'str' = 'default', field_c: 'int' = None) -> None>

### <font color='lilac'>Типизированные словари TypedDict</font>

- Схожими возможностями обладают _типизированные словари_ (`TypedDict`)

In [ ]:
from typing import TypedDict

class Data(TypedDict):
    int_field: int
    str_field: str

print(Data(int_field=10, str_field='string'))

- Использование классов данных является предпочтительнее по нескольким причинам:
    - классы поддерживают номинальную типизацию, а `TypedDict` - только структурную, т.е. `isinstance` использовать нельзя, нужно проверять соответствие полей во время выполнения кода, а это для больших структур может оказаться затратной процедурой
    - классы удобнее с точки зрения предоставляемого интерфейса (например, наличие properties)

### <font color='lilac'>Сторонние альтернативы классам данных</font>

- Использование классов данных не требует дополнительных зависимостей, что иногда может быть важным
- Но классы данных, как и прочие решения из стандартной библиотеки, не идеальны, например, не справляются с валидацией данных без написания громоздкого дополнительного кода

- Есть более развитые сторонние библиотеки для работы с типизированными данными, наиболее популярны `attrs` и `pydantic`
- На самом деле `attrs` появилась до Python 3.7, и классы данных создавались под её влиянием и с помощью её разработчиков
- [Ссылка](https://stefan.sofa-rockers.org/2020/05/29/attrs-dataclasses-pydantic) на хорошую статью про сравнение классов данных и библиотек `attrs` и `pydantic`
- Рассмотрим подробнее модуль `pydantic`, поскольку в связке с Fast API он особенно полезен при web-разработке

### <font color='lilac'>Модуль pydantic</font>

- Создание объектов производится наследованием от базового класса:

In [72]:
#!pip3 install pydantic

from typing import Optional, List
from pydantic import BaseModel

class Data(BaseModel):
    int_field: int
    str_field: str
    list_field: Optional[List[str]] = None

Data(**{'int_field': 10, 'str_field': 'string'})

Data(int_field=10, str_field='string', list_field=None)

- Типы входных данных автоматически валидируются:

In [ ]:
Data(**{'int_field': 'string', 'str_field': 'string'})

#ValidationError: 1 validation error for Data
#int_field
#  value is not a valid integer (type=type_error.integer)

- И даже исправляются, если преобразование известно:

In [74]:
Data(**{'int_field': '10', 'str_field': 'string'})

Data(int_field=10, str_field='string', list_field=None)

### <font color='lilac'>Вложенные pydantic-классы</font>

- Легко создавать и использовать рекурсивные модели данных с утиной типизацией:

In [75]:
from typing import List
from pydantic import BaseModel

class Foo(BaseModel):
    count: int
    size: float = None

class Bar(BaseModel):
    apple: str = 'x'
    banana: str = 'y'

class Spam(BaseModel):
    foo: Foo
    bars: List[Bar]

m = Spam(foo={'count': 4}, bars=[{'apple': 'x1'}, {'apple': 'x2'}])

print(m, '\n')
print(m.model_dump())

foo=Foo(count=4, size=None) bars=[Bar(apple='x1', banana='y'), Bar(apple='x2', banana='y')] 

{'foo': {'count': 4, 'size': None}, 'bars': [{'apple': 'x1', 'banana': 'y'}, {'apple': 'x2', 'banana': 'y'}]}


### <font color='lilac'>Валидация входных данных</font>

- Проверять можно не только типы, но значения входных данных:

In [ ]:
from pydantic import Field

class Data(BaseModel):
    str_field: str = Field(min_length=2, max_length=5)
    int_field: int = Field(le=150)

Data(str_field='', int_field=200)

#ValidationError: 2 validation errors for Data
#str_field
#  ensure this value has at least 2 characters (type=value_error.any_str.min_length; limit_value=2)
#int_field
#  ensure this value is less than or equal to 150 (type=value_error.number.not_le; limit_value=150)

- у типа `Field` есть много параметров для валидации, например:
    - `regex` : проверка соответствия строки заданному регулярному выражению
    - `multiple_of` : проверка того, что целое число является множителем заданного числа
    - `max_items` / `min_items` : проверка числа элементов в параметра-коллекциях
    - `allow_mutation` : разрешение или запрет изменения содержимого поля

### <font color='lilac'>Валидация входных данных</font>

- Для более сложных случаев описываются методы валидации:

In [ ]:
from pydantic import BaseModel, field_validator
from typing import Optional

class Data(BaseModel):
    str_field: str
        
    @field_validator('str_field')
    def str_field_validator(cls, value):
        if len(value) != 10:
            raise ValueError('Phone number must have 10 digits')
        return value

Data(str_field='99999999999')
#ValidationError: 1 validation error for Data
#str_field
#  Phone number must have 10 digits (type=value_error)

### <font color='lilac'>Дополнительные типы</font>

- `pydantic` предназначен для работы с реальными данными, поэтому в нём есть встроенные полезные типы:

In [80]:
from pydantic import (
    FilePath, HttpUrl, EmailStr, color,
    IPvAnyAddress, NegativeInt, PositiveFloat,
)

In [81]:
from pydantic import BaseModel, ValidationError

class Data(BaseModel):
    color: color.Color

print(Data(color='purple'))
print(Data(color='hsl(180, 100%, 50%)'))
print(Data(color='hsl(179, 100%, 50%)'))

#print(Data(color='hello')) ->

#ValidationError: 1 validation error for Data
#color
#  value is not a valid color: string not recognised as a valid color ...

color=Color('purple', rgb=(128, 0, 128))
color=Color('cyan', rgb=(0, 255, 255))
color=Color('#00fffb', rgb=(0, 255, 251))


### <font color='lilac'>Парсинг переменных окружения</font>

- `pydantic` позволяет парсить данные из файлов типа `.env` и напрямую приводить их к объектам типа `BaseSettings `
- Для этого нужно дополнительно установить модуль `python-dotenv`

In [ ]:
with open('.env', 'w') as fout:
    fout.write('''
        login=Login
        password=Password
    \n''')

In [82]:
#!pip3 install pydantic-settings
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    login: str
    password: str
    
    model_config = SettingsConfigDict(
        env_file='.env',
        env_file_encoding='utf-8'
    )

print(Settings())

login='Login' password='Password'


### <font color='lilac'>Генерация схемы данных</font>

In [83]:
import json
from enum import Enum
from pydantic import BaseModel, Field

class Counter(BaseModel):
    count: int
    size: float = None

class Gender(str, Enum):
    male = 'male'
    female = 'female'
    other = 'other'
    not_given = 'not_given'

class Model(BaseModel):
    counter: Counter = Field(...)
    gender: Gender = Field(None, alias='Gender')
    snap: int = Field(42, title='The Snap', gt=30, lt=50)

    class ConfigDict:
        title = 'Main'

print(json.dumps(Model.model_json_schema(), indent=2))

{
  "$defs": {
    "Counter": {
      "properties": {
        "count": {
          "title": "Count",
          "type": "integer"
        },
        "size": {
          "default": null,
          "title": "Size",
          "type": "number"
        }
      },
      "required": [
        "count"
      ],
      "title": "Counter",
      "type": "object"
    },
    "Gender": {
      "enum": [
        "male",
        "female",
        "other",
        "not_given"
      ],
      "title": "Gender",
      "type": "string"
    }
  },
  "properties": {
    "counter": {
      "$ref": "#/$defs/Counter"
    },
    "Gender": {
      "$ref": "#/$defs/Gender",
      "default": null
    },
    "snap": {
      "default": 42,
      "exclusiveMaximum": 50,
      "exclusiveMinimum": 30,
      "title": "The Snap",
      "type": "integer"
    }
  },
  "required": [
    "counter"
  ],
  "title": "Model",
  "type": "object"
}


### <font color='lilac'>Декораторы</font>

- Декораторы в Python - это инструмент языка, предназначенный для добавления новых свойств функциям, классам и методам на этапе их определения
- В Python это особенно просто за счёт duck-typing
- Этот тот случай, когда нужно вспомнить о замыканиях (функциях, возвращающих функции)
- Пример: декоратор, замеряющий время работы функции

In [84]:
def timed(callable_obj):
    import time

    def __timed(*args, **kw):
        time_start = time.time()
        result = callable_obj(*args, **kw)
        time_end = time.time()
        
        print('{}  {:.3f} ms'.format(callable_obj.__name__,
                                     (time_end - time_start) * 1000))
        return result

    return __timed

In [87]:
@timed
def func():
    for i in range(1000000):
        pass
func()

func  16.978 ms


- На самом деле функция выглядит так:

In [88]:
import inspect
lines = inspect.getsource(func)
print(lines)

    def __timed(*args, **kw):
        time_start = time.time()
        result = callable_obj(*args, **kw)
        time_end = time.time()

        print('{}  {:.3f} ms'.format(callable_obj.__name__,
                                     (time_end - time_start) * 1000))
        return result



### <font color='lilac'>Корректное определение декоратора</font>

- Желательно, чтобы декорированная функция сохранила в себе информацию о своем исходном коде

In [89]:
from functools import wraps

def timed(callable_obj):
    import time

    @wraps(callable_obj)
    def __timed(*args, **kw):
        time_start = time.time()
        result = callable_obj(*args, **kw)
        time_end = time.time()
        
        print('{}  {:.3f} ms'.format(callable_obj.__name__,
                                     (time_end - time_start) * 1000))
        return result

    return __timed

In [90]:
@timed
def func():
    for i in range(1000000):
        pass

print(inspect.getsource(func))

@timed
def func():
    for i in range(1000000):
        pass



### <font color='lilac'>Декоратор с параметрами и состоянием</font>

- Короткие декораторы без внутреннего состояния можно описывать с помощью функций
- В более сложных случаях использование связки "класс+функция" даёт более понятный код

In [92]:
import time

class TimedDecoratorImpl:
    def __init__(self, callable_obj, format_str):
        self.callable_obj = callable_obj
        self.format_str = format_str

    def __call__(self, *args, **kwargs):
        time_start = time.time()
        result = self.callable_obj(*args, **kwargs)
        time_end = time.time()
        
        print(self.format_str.format(self.callable_obj.__name__, (time_end - time_start) * 1000))
        return result

def timed(format_str):
    def decorator(callable_obj):
        return TimedDecoratorImpl(callable_obj, format_str)
    return decorator

In [95]:
@timed(format_str='{}  {:.3f} ms')
def func():
    for i in range(1000000):
        pass

# print(inspect.getsource(func))
func()

func  16.667 ms


- Но с классами нельзя использовать `functools.wraps`, альтернатива такая:

In [96]:
def timed(format_str):
    import time

    def decorator(callable_obj):

        @wraps(callable_obj)
        def wraper_func(*args, **kwargs):
            time_start = time.time()
            result = callable_obj(*args, **kwargs)
            time_end = time.time()
            
            print(format_str.format(callable_obj.__name__, (time_end - time_start) * 1000))
            return result

        return wraper_func
    return decorator

In [97]:
@timed(format_str='{}  {:.3f} ms')
def func():
    for i in range(1000000):
        pass

print(inspect.getsource(func))
func()

@timed(format_str='{}  {:.3f} ms')
def func():
    for i in range(1000000):
        pass

func  18.961 ms


### <font color='lilac'>Декораторы классов</font>

- Целые классы тоже може декорировать, как и функции/методы
- Пример: декоратор класса, который получает на вход декоратор функции и оборачивает его вокруг каждого публичного метода класса

In [98]:
def decorate_class(decorator, *args, **kwargs):
    def _decorate(cls):
        for f in cls.__dict__:
            if callable(getattr(cls, f)) and not f.startswith("_"):
                setattr(
                    cls,
                    f,
                    # 1: run outer timed func to get decorator parameters
                    # 2: run middle decorator with these parameters
                    # 3: get final decorated (wraped) function for further calling with its params
                    decorator(*args, **kwargs)(getattr(cls, f))
                )
        return cls
    return _decorate

In [99]:
@decorate_class(timed, format_str='{}  {:.3f} ms')
class Cls:
    a = 10
    def method(self): time.sleep(1)
    def _method_2(self): time.sleep(2)

Cls().method()
Cls().a  # not callable
Cls()._method_2()  # not public

method  1000.618 ms


In [100]:
print(inspect.getsource(Cls().method))

    def method(self): time.sleep(1)



- При определении класса производится вызов `decorate_class._decorate`, которая возвращает обновлённый класс